This notebook can be used to generate the main graphs in our paper on mechanotransduction signaling in cells on nanopillars.

First, we introduce the necessary imports and configure our matplotlib settings.

In [ ]:
import sys, os, pathlib
sys.path.append("/root/shared/gitrepos/smart-mechanotransduction/utils")
sys.path.append("/root/shared/gitrepos/smart-comp-sci/mechanotransduction-example")
cur_dir = str(pathlib.Path.cwd() / "..")
import smart_analysis
from mechanotransduction_ode import mechanotransduction_ode_calc
from matplotlib import pyplot as plt
import matplotlib
plt.style.use(f"smart_plots.mplstyle")
matplotlib.rcParams.update({'figure.figsize': (5,4)})
import numpy as np

This runs data analysis on all simulations in all subfolders within the folder specified by `files_dir` and saves into `npy_dir`. The correct mesh must be included in a subfolder of `mesh_dir`.

In [ ]:
import re
mesh_dir = "/root/shared/gitrepos/smart-nanopillars/meshes/nanopillars_baseline"
files_dir = "/root/scratch/nuc_mechanics_data/results_nanopillars_sweepForce_2026-01-01_scalept2"
npy_dir = pathlib.Path.cwd() / ".." / "analysis_data" / "npy-files-sweepForce_newPhos_scalept2"
npy_dir.mkdir(exist_ok=True)
test_folders = os.listdir(files_dir)
condition_str = []
for i in range(len(test_folders)):
    mesh_file = f"{files_dir}/{test_folders[i]}/mesh/nuc_mesh.h5"
    # for folder in os.listdir(mesh_dir):
    #     if folder[12:] in test_folders[i]: # mesh name matches current case
    #         mesh_file = f"{mesh_dir}/{folder}/spreadCell_mesh.h5"
    #         break
    # if mesh_file == "":
    #     print("Mesh could not be found, skipping to next case")
    #     condition_str.append("")
    #     continue
    results_folder = f"{files_dir}/{test_folders[i]}"
    condition_cur = test_folders[i]
    condition_str.append(condition_cur)
    if mesh_file=="" or results_folder=="":
        raise ValueError("Folders do not match expected structure for analysis")
    height = 3.0
    zmax_nuc = 6.4
    if "loc" in test_folders[i]:
        loc = float(re.findall(r"loc.*?_", test_folders[i])[0][3:-1])
    else:
        loc = 0
    tests = ["all"]#,"center_low","center_high"] # other spatial domains could be specified here
    center_low = [loc-1, -1, 0, loc+1, 1, height+1]
    center_high = [loc-1, -1, zmax_nuc-1.0, loc+1, 1, zmax_nuc+10]
    domains = [[], center_low, center_high]
    for sd in range(len(tests)):
        # try:
        tVec, results_all = smart_analysis.analyze_all(
                mesh_file=mesh_file, results_path=results_folder, display=False, axisymm=False,
                subdomain=domains[sd])
        # except:
        #     print("error in analysis, skipping to next case")
        #     continue
        results_all.insert(0, tVec) # add time as first element in list
        max_length = len(tVec)
        for j in range(len(results_all)):
            if len(results_all[j]) > max_length:
                max_length = len(results_all[j])
        for j in range(len(results_all)):
            num_zeros = max_length - len(results_all[j])
            for k in range(num_zeros):
                results_all[j].append(0)
        np.save(npy_dir / f"{condition_cur}_results_{tests[sd]}.npy", results_all)

Generate graphs of YAP/TAZ N/C over time for cells on different nanopillar substrates (Fig 2D) and store relevant steady-state concentrations.

In [ ]:
from smart import mesh_tools
import dolfin as d

# laminDiff = 0.001 # or .01
# pitchArray=[0.0, 1.5, 3.0, 1.5, 3.0]
# radiusArray=[0.0, 0.2, 0.2, 0.5, 0.5]
# forceArray=[0.0, 400.0, 400.0, 800.0, 800.0, 0.0, 400.0, 400.0, 800.0, 800.0]
# t0Array=[100.0, 100.0, 1000.0, 100.0, 1000.0, 100.0, 100.0, 1000.0, 100.0, 1000.0]
# actinArray=[0.0, 0.0, 0.0, 0.0, 0.0, 20.0, 20.0, 20.0, 20.0, 20.0]
pitchArray = [0.0, 3.0, 3.0]
radiusArray = [0.0, 0.2, 0.5]
substrate_names = ["flat", "200 nm NP", "500 nm NP"]
forceArray = [0.0, 100.0, 200.0, 300.0, 400.0, 500.0, 600.0, 700.0, 800.0]
var_names_saved = ["aNPC", "FActin", "J_NE", "J_Nuc", "LaminA", "LaminAp", "NE_alt_udef", "NE_udef", 
                   "NPC", "NPC_A", "Nuc_udef", "rupture_marker", "YAPTAZ_nuc"]
# var_names_saved = var_names_saved[1:]
plot_names = ["YAPTAZ_nuc", "LaminA", "LaminAp", "rupture_marker"]
results_idx = []
for name in plot_names:
    results_idx.append(var_names_saved.index(name) + 1) # add one because time is first el

YAPTAZ_ratios = np.zeros([len(pitchArray),len(forceArray)])
vol_vals = np.zeros([len(pitchArray),len(forceArray)])
for phiE in [False]:#[True,False]:
    for i in [0]:#range(len(pitchArray)):
        for j in range(len(forceArray)):
            if phiE:
                cur_folder = (f"/root/scratch/nuc_mechanics_data/results_nanopillars_sweepForce_2026-01-01_scalept2_phiE/"
                            f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}")#_t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}")
                file_cur = (f"{cur_dir}/analysis_data/npy-files-sweepForce_newPhos_scalept2_phiE/"#nuc-mechanics-coupled-corrected/"
                    f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                    f"force{forceArray[j]}_results_all.npy")#t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}_results_all.npy")
            else:
                cur_folder = (f"/root/scratch/nuc_mechanics_data/results_nanopillars_sweepForce_2026-01-01_scalept2/"
                            f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}")#_t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}")
                file_cur = (f"{cur_dir}/analysis_data/npy-files-sweepForce_newPhos_scalept2/"#nuc-mechanics-coupled-corrected/"
                    f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                    f"force{forceArray[j]}_results_all.npy")#t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}_results_all.npy")
            loaded_cur = mesh_tools.load_mesh(pathlib.Path(f"{cur_folder}/mesh/nuc_mesh.h5"))
            YAPTAZ_cur = np.loadtxt(pathlib.Path(f"{cur_folder}/YAPTAZ.txt"))
            vol_inner = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/vol_inner.txt"))
            
            results_cur = np.load(file_cur)
            YAPnuc = results_cur[results_idx[0]]
            if radiusArray[i] == 0.0:
                FActin0 = 140
            elif radiusArray[i] <= 0.15:
                FActin0 = 80.0
            elif radiusArray[i] <= 0.25:
                FActin0 = 100.0
            elif radiusArray[i] <= 0.5:
                FActin0 = 120.0
            else:
                FActin0 = 140.0
            Myo_A = 4.0
            K_CN = 4.0
            K_CY = 5.429e-3
            phiNP = (K_CN + K_CY*FActin0*Myo_A)/(1+ K_CN + K_CY*FActin0*Myo_A)
            YAPcyto = YAPTAZ_cur[:,1]/phiNP
            maxIdx = min([len(YAPnuc),len(YAPcyto)])
            YAPratio = YAPnuc[0:maxIdx] / YAPcyto[0:maxIdx]
            LaminA = results_cur[results_idx[1]]
            YAPTAZ_ratios[i,j] = YAPratio[-1]
            vol_vals[i,j] = vol_inner[-1]
            # plt.plot(results_cur[0], YAPratio,
            #         label=f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_force{forceArray[j]}")#_t0{t0Array[j]}_actin{actinArray[j]}")
        plt.plot(vol_vals[i,:]/max(vol_vals[i,:]), YAPTAZ_ratios[i,:], 'o', label=substrate_names[i])
        # plt.plot(forceArray, vol_vals[i,:], 'o', label=substrate_names[i])

# plt.legend()
# plt.ylabel("YAP/TAZ N/C")
# plt.xlabel('Time (s)')
# plt.xlim([0, 3600])
koushki_data = np.loadtxt("koushki_data.txt")
plt.plot(koushki_data[:,0]/2136.114733, koushki_data[:,2], 'x', label="Koushki et al 2023")
plt.xlim([0.3, 1.1])
plt.xlabel('Relative volume')
plt.ylabel('YAP/TAZ N/C')
plt.legend()
# plt.savefig("YTdyn.pdf", format="pdf")

In [ ]:
from smart import mesh_tools
import dolfin as d
import h5py

pitchArray=[0.0, 2.0, 3.0, 4.0, 5.0, 6.0]
radiusArray=[0.0, 0.2, 0.2, 0.2, 0.2, 0.2]
forceArray=[0.0, 400.0, 400.0, 800.0, 800.0]
t0Array=[100.0, 100.0, 1000.0, 100.0, 1000.0]
diffArray=[0.001, 0.001, 0.01, 0.01]
actinArray=[0.0, 20.0, 0.0, 20.0]
noCycle = False
parent_folder = "/root/scratch/nuc_mechanics_data/results_nanopillars_2025-12-21"
if noCycle:
    parent_folder = f"{parent_folder}_noCycle"

var_names_saved = ["aNPC", "FActin", "J_NE", "J_Nuc", "LaminA", "LaminAp", "NE_alt_udef", "NE_udef", 
                   "NPC", "NPC_A", "Nuc_udef", "rupture_marker", "YAPTAZ_nuc"]
# var_names_saved = var_names_saved[1:]
plot_names = ["YAPTAZ_nuc", "LaminA", "LaminAp", "rupture_marker", "NPC_A"]
results_idx = []
for name in plot_names:
    results_idx.append(var_names_saved.index(name) + 1) # add one because time is first el

YAPTAZ_ratios = np.zeros([len(pitchArray),len(forceArray)])
vol_vals = np.zeros([len(pitchArray),len(forceArray)])
for i in [0,1,2,3,4,5]:#range(len(pitchArray)):
    for j in [4]:#range(len(forceArray)):
        for k in [3]:#range(len(diffArray)):
            cur_folder = (f"{parent_folder}/nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                        f"force{forceArray[j]}_t0{t0Array[j]}_diff{diffArray[k]}_actin{actinArray[k]}")
            loaded_cur = mesh_tools.load_mesh(pathlib.Path(f"{cur_folder}/mesh/nuc_mesh.h5"))
            YAPTAZ_cur = np.loadtxt(pathlib.Path(f"{cur_folder}/YAPTAZ.txt"))
            vol_inner = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/vol_inner.txt"))
            file_cur = (f"{cur_dir}/analysis_data/npy-files-2025-12-21/"#nuc-mechanics-coupled-corrected/"
                        f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                        f"force{forceArray[j]}_t0{t0Array[j]}_diff{diffArray[k]}_actin{actinArray[k]}_results_all.npy")
            results_cur = np.load(file_cur)
            YAPnuc = results_cur[results_idx[0]]
            if radiusArray[i] == 0.0:
                FActin0 = 140
            elif radiusArray[i] <= 0.15:
                FActin0 = 80.0
            elif radiusArray[i] <= 0.25:
                FActin0 = 100.0
            elif radiusArray[i] <= 0.5:
                FActin0 = 120.0
            else:
                FActin0 = 140.0
            Myo_A = 4.0
            K_CN = 4.0
            K_CY = 5.429e-3
            phiNP = (K_CN + K_CY*FActin0*Myo_A)/(1+ K_CN + K_CY*FActin0*Myo_A)
            YAPcyto = YAPTAZ_cur[:,1]/phiNP
            maxIdx = min([len(YAPnuc),len(YAPcyto)])
            YAPratio = YAPnuc[0:maxIdx] / YAPcyto[0:maxIdx]
            LaminA = results_cur[results_idx[1]]
            YAPTAZ_ratios[i,j] = YAPratio[-1]
            vol_vals[i,j] = vol_inner[-1]
            # rupture_marker_vals = []
            # rupture_file = h5py.File(f"{cur_folder}/rupture_marker.h5", 'r')
            # for idx in range(len(vol_inner)-1):
            #     try:
            #         rupture_array = rupture_file['VisualisationVector'][f'{idx}'][()]
            #     except:
            #         break
            #     rupture_marker_vals.append(np.max(rupture_array))
            plt.plot(results_cur[0], YAPratio,
                    label=f"{i},{j},{k}")
            # plt.plot(rupture_marker_vals)
    # plt.plot(vol_vals[i,:]/max(vol_vals[i,:]), YAPTAZ_ratios[i,:], 'o', label=substrate_names[i])

plt.legend()
plt.ylabel("Rupture vals")
plt.xlabel('Time (s)')
# plt.xlim([0, 3600])
# plt.savefig("YTdyn.pdf", format="pdf")

In [ ]:
var_names_saved = ["Cofilin_NP", "FActin", "GActin", "integrin", "integrin_L", "integrin_LT", "integrin_LTst", "integrin_tot",
                   "LaminA", "ligand", "LIMK_A", "mDia_A", "Myo_A", "NPC_A", "pFAK", "RhoA_GDP", "RhoA_GTP_Contact", "RhoA_GTP_PM", 
                   "ROCK_A", "YAPTAZ", "YAPTAZ_nuc", "YAPTAZ_phos"]
plot_names = ["YAPTAZ_phos", "YAPTAZ", "YAPTAZ_nuc"]
source_folder = "/root/shared/gitrepos/smart-mechanotransduction/analysis_data/npy-files-nuc-mechanics-coupled"
folder_list = os.listdir(source_folder)

for i in range(1,len(folder_list)):#[7, 10, 11, 12, 13, 14]:
    results_cur = np.load(f"{source_folder}/{folder_list[i]}")
    YAPTAZ_nuc = results_cur[10]
    # YAPratio_SS[i][j] = YAPratio[-1]
    # if i == 10:# and np.isclose(wasp_val, 0.01):
    if "p3.0" in folder_list[i]:
        plt.plot(results_cur[0], YAPTAZ_nuc, label=folder_list[i], linestyle="dashed")
    else:
        plt.plot(results_cur[0], YAPTAZ_nuc, label=folder_list[i])

# plt.legend()
plt.ylabel("Nuclear YAP/TAZ")
plt.xlabel('Time (s)')
plt.xlim([0, 1800])
# plt.savefig("cytosolic activation.pdf", format="pdf")

Plot relationship between traction stress and substrate stiffness.

In [ ]:
alphaSt = 0.1
forceMax = 30; # pN per molecule
Evals = np.logspace(-1, 3, 101)
tractionForce = forceMax*(0.2 + 0.8*np.exp(-((np.log(Evals)-np.log(10))/2)**2))
plt.semilogx(Evals, tractionForce)
plt.ylim([0, 32])
plt.xlabel('Substrate stiffness (kPa)')
plt.ylabel('Force per integrin (pN)')
plt.savefig("traction.pdf")

Plot summary of cytosolic stiffness, stress fibers, and pFAK at steady state (Fig 1C-F)

In [ ]:
vars = [9e-6*FActin_SS**2.6, FActin_SS*MyoA_SS/548.444, pFAK_SS, YAPratio_SS]#, ILTst_SS]
names = ["Cyto stiffness", "Stress fibers", "pFAK", "YAP NC"]#, "Stretch integrin"]
for i in range(len(vars)):
    plt.figure()
    cur_var = vars[i]
    name = names[i]
    width = 0.75  # the width of the bars

    x = [0,1,2,3,4,5]
    fig, ax = plt.subplots(layout='constrained', figsize=(5,3))
    ax.bar(x, cur_var[[14,13,12,11,10,7],0], width)
    ax.set_ylabel(name)
    ax.set_xticks(x, ['1 kPa','3 kPa','7 kPa', '14 kPa', 'glass', '250 nm NP'])
    if name == "YAP NC":
        plt.ylim([0.75, 0.95])
    plt.savefig(f"{name}.pdf")

Plot NSSE as a function of alpha0 (curvature sensitivity parameter):

In [ ]:
curvSens = [0, 1/10, 1/5, 1/2, 1]
errorsSel0 = [np.sum(errors_list[0][:,0]), np.sum(errors_list[0][:,4]), np.sum(errors_list[0][:,3]),
            np.sum(errors_list[0][:,2]), np.sum(errors_list[0][:,1])]
plt.plot(curvSens, errorsSel0, 'o-', label="kWASP=0.001")
errorsSel1 = [np.sum(errors_list[1][:,0]), np.sum(errors_list[1][:,4]), np.sum(errors_list[1][:,3]),
            np.sum(errors_list[1][:,2]), np.sum(errors_list[1][:,1])]
plt.plot(curvSens, errorsSel1, 'o--', label="kWASP=0.01")
# plt.ylim([100,300])
plt.ylabel("Normalized SSE")
plt.xlabel("Curvature sensitivity")
plt.legend()
# plt.savefig("curv_sens_fit.pdf", format="pdf")

Compare simulation YAP/TAZ N/C to experimentally measured values. If `multCond` is true, then plot for different levels of curvature sensitivity (Fig S1), otherwise create graph shown in Fig 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
multCond = False

conditions = []
for i in [10,0, 1, 2, 7, 6]:
    conditions.append(f"h{heightArray[i]}p{pitchArray[i]}r{radiusArray[i]}")
YAPTAZ_expt = np.array(yapData)
YAPTAZ_expt_std = np.array(yapStdData)
if multCond:
    np_means = {
        'Experiments': YAPTAZ_expt[[10,0,1,2,7,6]],
        'No curv sens': YAPTAZ_ratios[[10,0,1,2,7,6],0],
        'H0=10': YAPTAZ_ratios[[10,0,1,2,7,6],4],
        'H0=5': YAPTAZ_ratios[[10,0,1,2,7,6],3],
        'H0=2': YAPTAZ_ratios[[10,0,1,2,7,6], 2],
    }
    width = 0.12  # the width of the bars
else:
    np_means = {
        'Experiments': YAPTAZ_expt[[10,0,1,2,7,6]],
        'H0=5': YAPTAZ_ratios[[10,0,1,2,7,6],3],
    }
    width = 0.3

x = np.arange(len(conditions))  # the label locations
multiplier = 0

fig, ax = plt.subplots(layout='constrained', figsize=(6,4))

for attribute, measurement in np_means.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    # ax.bar_label(rects, padding=3)
    multiplier += 1

ax.errorbar(x, YAPTAZ_expt[[10,0,1,2,7,6]], YAPTAZ_expt_std[[10,0,1,2,7,6]], linestyle='None', color="r", capsize=4)

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('YAP/TAZ N/C')
ax.set_xticks(x + width/2, conditions)
ax.legend()#loc='upper right')
plt.ylim([0.0, 4.5])
# plt.savefig("bargraph_multcond.pdf", format="pdf")

Create plots for different vertical positions of nuclear (Fig S2)

In [ ]:
moveNucArray = [0.0, -0.6, -1.0, -1.4]
var_names_saved = ["Cofilin_NP", "FActin", "GActin", "LaminA", "LIMK_A", "mDia_A", 
                   "Myo_A", "NPC_A", "pFAK", "RhoA_GDP", "RhoA_GTP", "ROCK_A", 
                   "Substrate_A", "YAPTAZ", "YAPTAZ_nuc", "YAPTAZ_phos"]
plot_names = ["YAPTAZ_phos", "YAPTAZ", "YAPTAZ_nuc"]
results_idx = []
for name in plot_names:
    results_idx.append(var_names_saved.index(name) + 1) # add one because time is first el
fig, ax = plt.subplots()

YAPTAZ_ratios = np.zeros([len(moveNucArray), 1])

for i in range(len(moveNucArray)):
    file_cur = (f"{cur_dir}/analysis_data/npy-files-nanopillars-movenuc"
                    f"/nanopillars_movenuc{moveNucArray[i]}_results_all.npy")
    try:
        results_cur = np.load(file_cur)
    except:
        continue
    YAPphos = results_cur[results_idx[0]]
    YAPnp = results_cur[results_idx[1]]
    YAPnuc = results_cur[results_idx[2]]
    YAPratio = YAPnuc / (YAPphos + YAPnp)
    YAPTAZ_ratios[i] = YAPratio[-1]
    plt.plot(results_cur[0], YAPratio,
        label=f"movenuc{moveNucArray[i]}")
plt.ylabel("YAP/TAZ N/C")
plt.xlabel('Time (s)')
plt.legend()
plt.xlim([0, 3600])
# plt.ylim([2.9, 3.2])
# plt.savefig("YAPTAZ_movenuc.pdf", format="pdf")

Plot NPC import rate as a function of stretch for different stretch sensitivites (Fig 4E).

In [ ]:
stretch_vals = np.linspace(0.1, 10, 200)
plt.loglog(stretch_vals, np.ones_like(stretch_vals),label='no stretch sens.')
plt.loglog(stretch_vals, np.exp((stretch_vals-1)/2.5),label='high stretch sens.')
plt.loglog(stretch_vals, np.exp((stretch_vals-1)/5.0), label='moderate stretch sens.')
plt.loglog(stretch_vals, np.exp((stretch_vals-1)/7.5), label='low stretch sens.')

plt.xlabel('Stretch ratio')
plt.ylabel('Relative import rate')
plt.legend()
# plt.savefig("/root/shared/gitrepos/smart-comp-sci-data/pyplots/nanopillars/stretch_transport.pdf", format="pdf")

Plot YAP/TAZ N/C for different stretch sensitivities, store SS YAP/TAZ N/C.

In [ ]:
indentationArray = [0.0, 0.4, 0.8, 
                    1.0, 1.4, 1.8, 
                    2.0, 2.4, 2.8]
a0Array=[0, 2.5, 5.0, 7.5]
var_names_saved = ["Cofilin_NP", "FActin", "GActin", "LaminA", "LIMK_A", "mDia_A", 
                   "Myo_A", "NPC_A", "pFAK", "RhoA_GDP", "RhoA_GTP", "ROCK_A", 
                   "Substrate_A", "YAPTAZ", "YAPTAZ_nuc", "YAPTAZ_phos"]
plot_name = "FActin"
results_idx = var_names_saved.index(plot_name) + 1 # add one because time is first el
plot_names = ["YAPTAZ_phos", "YAPTAZ", "YAPTAZ_nuc"]
results_idx = []
for name in plot_names:
    results_idx.append(var_names_saved.index(name) + 1) # add one because time is first el

fig, ax = plt.subplots()

YAPTAZ_ratios = np.zeros([len(a0Array), len(indentationArray)])

for i in range(len(a0Array)):
    for j in range(len(indentationArray)):
        for domain in ["all"]:
            file_cur = (f"{cur_dir}/analysis_data/npy-files-nanopillars-withwasp-indent"
                    f"/nanopillars_indent{indentationArray[j]}_a0_{a0Array[i]}_results_{domain}.npy")
            try:
                results_cur = np.load(file_cur)
            except:
                continue
            if a0Array[i] > 0 and indentationArray[j] > 0: # then additional a0 variable included 
                YAPphos = results_cur[results_idx[0]+1]
                YAPnp = results_cur[results_idx[1]+1]
                YAPnuc = results_cur[results_idx[2]+1]
            else:
                YAPphos = results_cur[results_idx[0]]
                YAPnp = results_cur[results_idx[1]]
                YAPnuc = results_cur[results_idx[2]]
            YAPratio = YAPnuc / (YAPphos + YAPnp)
            YAPTAZ_ratios[i][j] = YAPratio[-1]
            if a0Array[i] == 5.0 and indentationArray[j] == 2.8:
                YAPTAZ_ratio_ref = YAPratio[-1]
                YAPTAZ_dyn_ref = [results_cur[0], YAPratio]
                NPC_ref = results_cur[8][-1]
                phiRef = YAPphos[-1]/(YAPphos[-1] + YAPnp[-1])
            if domain=="center_low":
                plt.plot(results_cur[0], YAPratio,
                    label=f"indent{indentationArray[j]}_a0_{a0Array[i]} low", linestyle="dashed")
            else:
                plt.plot(results_cur[0], YAPratio,
                label=f"indent{indentationArray[j]}_a0_{a0Array[i]}")
# plt.legend()
plt.ylabel("YAP/TAZ N/C")
plt.xlabel('Time (s)')
plt.legend()
plt.xlim([0, 3600])
# plt.ylim([2.9, 3.2])
# plt.savefig("YAPTAZ_withcompression.pdf", format="pdf")

Plot SS YAP/TAZ as a function of indentation for different stretch sensitivities.

In [ ]:
selIdx1 = np.array([0,1,3,4,5,6,7,8])
plt.plot(np.array(indentationArray), YAPTAZ_ratios[0], marker="o",label="no stretch sensitivity")
plt.plot(indentationArray, YAPTAZ_ratios[1], marker="o", label="with highly stretch sensitive NPCs")
plt.plot(indentationArray, YAPTAZ_ratios[2], marker="o", label="with moderately stretch sensitive NPCs")
plt.plot(indentationArray, YAPTAZ_ratios[3], marker="o", label="with low stretch sensitive NPCs")
# plt.ylim([2, 3])
# plt.xlim([-.05, 1.55])
plt.xlabel("Indentation (μm)")
plt.ylabel("YAP/TAZ N/C")
plt.legend()
# plt.savefig("indentation_NCsummary.pdf", format="pdf")

Create disk mesh to use for projection of F-actin and Lamin A steady state data.

In [ ]:
rad = 6.0
import gmsh
import numpy as np
import pathlib
from smart import mesh_tools
gmsh.initialize()
gmsh.model.add("arc")
surf = gmsh.model.occ.add_disk(0, 0, 0, rad, rad)
gmsh.model.occ.synchronize()
gmsh.model.add_physical_group(2, [surf], tag=1)
facets = gmsh.model.getBoundary([(2, surf)])
gmsh.model.add_physical_group(1, [facets[0][1]], tag=10)
def meshSizeCallback(dim, tag, x, y, z, lc):
    return 0.15

gmsh.model.mesh.setSizeCallback(meshSizeCallback)
# set off the other options for mesh size determination
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
# this changes the algorithm from Frontal-Delaunay to Delaunay,
# which may provide better results when there are larger gradients in mesh size
gmsh.option.setNumber("Mesh.Algorithm", 5)
gmsh.model.mesh.generate(3)
tmp_folder = pathlib.Path(f"tmp_ellipse")
tmp_folder.mkdir(exist_ok=True)
gmsh_file = tmp_folder / "ellipses.msh"
gmsh.write(str(gmsh_file))
gmsh.finalize()
dmesh, mf1, mf2 = mesh_tools.gmsh_to_dolfin(str(gmsh_file), tmp_folder, 2)
# remove tmp mesh and tmp folder
gmsh_file.unlink(missing_ok=False)
tmp_folder.rmdir()

Compute projection of FActin and LaminA at the z = 2 plane at steady-state. First need to compute the interpolation of 2.8 micron indentation by running interp_meshes.

In [ ]:
import sys
import numpy as np
import dolfin as d
from matplotlib import pyplot as plt
sys.path.append("/root/shared/gitrepos/smart-nanopillars/utils")
from smart_analysis import load_solution
mesh_parent_folder = f"{cur_dir}/meshes"
results_folder = f"{cur_dir}/analysis_data/simulation_results_2.8indent_interp"
stress_fiber_int = []
NPC_int = []
idx = 500
for sol in load_solution(mesh_file=f"{mesh_parent_folder}/fullMesh2.8/spreadCell_mesh.h5", 
                            results_file=[f"{results_folder}/FActin_interp.h5", 
                                          f"{results_folder}/LaminA_interp.h5"],
                            idx=idx):
    FActin, LaminA = sol[:]
    dxCyto = d.Measure("dx", FActin.function_space().mesh())
    dxNE = d.Measure("dx", LaminA.function_space().mesh())
    CytoSubmesh = dmesh
    CytoCoordsRef = CytoSubmesh.coordinates()
    VFlat = d.FunctionSpace(CytoSubmesh, "P", 1)
    FActinProj = d.Function(VFlat)
    FActinProjVec = FActinProj.vector().get_local()
    LaminAProj = d.Function(VFlat)
    LaminAProjVec = FActinProj.vector().get_local()
    CytoCoords = VFlat.tabulate_dof_coordinates()
    multFactor = 1/(0.4*0.4*2*0.4*np.pi**(3/2))
    for i in range(len(CytoCoords)):
        curKernel = d.Expression(f"{multFactor}*exp(-(pow(x[0]-({CytoCoords[i,0]}),2) + "
                                 f"pow(x[1]-({CytoCoords[i,1]}),2) + pow((x[2]-2.0)/2,2))/0.16)", 
                                 degree=1)
        FActinProjVec[i] = d.assemble(curKernel*FActin*dxCyto)
        LaminAProjVec[i] = d.assemble(curKernel*LaminA*dxNE)
        print(f"{i+1} of {len(CytoCoords)}")
    FActinProj.vector().set_local(FActinProjVec)
    FActinProj.vector().apply("insert")
    LaminAProj.vector().set_local(LaminAProjVec)
    LaminAProj.vector().apply("insert")
    d.File("FActinProj.pvd") << FActinProj
    d.File("LaminAProj.pvd") << LaminAProj
    break

Plot YAP/TAZ N/C for cases of nuclear rupture (Fig 5E). Must run cases of stretch sensitivity above first to load reference YAP/TAZ data.

In [ ]:
indentation = 2.8#[0.8, 1.8, 2.8]
poreSizeArray=[0.1, 0.2]#, 0.1, 0.2, 0.5]
poreRateArray=[1]
transportRateArray=[10000, 10000, 10000, 10000]#, 100, 100, 10000, 10000, 10000]
transportRatioArray=[1, 5, 1, 5]

var_names_saved = ["YAPTAZ", "YAPTAZ_nuc", "YAPTAZ_nuc_phos", "YAPTAZ_phos"]
plot_names = ["YAPTAZ_phos", "YAPTAZ", "YAPTAZ_nuc", "YAPTAZ_nuc_phos"]
results_idx = []
for name in plot_names:
    results_idx.append(var_names_saved.index(name) + 1) # add one because time is first el
fig, ax = plt.subplots()

pore_dir = f"{cur_dir}/analysis_data/npy-files-nanopillars-withwasp-pores"

YAPTAZ_ratios = np.zeros([len(poreSizeArray),len(poreRateArray),len(transportRateArray)])

for i in range(len(poreSizeArray)):
    for j in range(len(poreRateArray)):
        for k in range(len(transportRateArray)):
            cur_file = (f"nanopillars_indent{indentation}_pore_size{poreSizeArray[i]}_"
                    f"loc0.0_rate{poreRateArray[j]}_transport{transportRateArray[k]}_"
                    f"ratio{transportRatioArray[k]}_results_all.npy")
            cur_file = f"{pore_dir}/{cur_file}"
            try:
                results_cur = np.load(cur_file)
            except:
                continue
            if results_cur[0][-1] < 1e5:
                print("short")
            YAPphos = results_cur[results_idx[0]]
            YAPnp = results_cur[results_idx[1]]
            YAPnuc = results_cur[results_idx[2]]
            YAPphos_nuc = results_cur[results_idx[3]]
            YAPratio = (YAPnuc + YAPphos_nuc) / (YAPphos + YAPnp)
            plt.plot(results_cur[0], YAPratio, label=f"Pore size = {poreSizeArray[i]}")
            YAPTAZ_ratios[i][j][k] = YAPratio[-1]
plt.plot(YAPTAZ_dyn_ref[0], YAPTAZ_dyn_ref[1], linestyle="dashed", label="no pore")
plt.legend()
plt.ylabel("YAP/TAZ N/C")
plt.xlabel('Time (s)')
plt.xlim([0, 3600])

Write spatiotemporal dynamics of pore formation to file. Currently runs on the full mesh generated after running `interp_meshes`, but can also run on other meshes (change path to mesh accordingly).

In [ ]:
import dolfin as d
from smart import mesh_tools
import pathlib
import numpy as np
loaded = mesh_tools.load_mesh(pathlib.Path(f"{cur_dir}/meshes/fullMesh2.8") / "spreadCell_mesh.h5")
mesh_cur = d.create_meshview(loaded.mf_facet, 12)
Vcur = d.FunctionSpace(mesh_cur, "P", 1)
xShift = 0.0
centerLogic = np.logical_and(np.isclose(mesh_cur.coordinates()[:,0], xShift, 0.1),
               np.logical_and(
               np.isclose(mesh_cur.coordinates()[:,1], 0.),
               mesh_cur.coordinates()[:,2] < 0.8*np.max(mesh_cur.coordinates()[:,2])))
centerIdx = np.nonzero(centerLogic)[0]
centerIdx = centerIdx[np.argmin(np.abs(mesh_cur.coordinates()[centerIdx,0]-xShift))]
zCenter = mesh_cur.coordinates()[centerIdx,2]
tVals = np.linspace(0., 5., 200)
pore_xdmf = d.XDMFFile(d.MPI.comm_world, "pore.xdmf")
pore_xdmf.parameters["flush_output"] = True
t = d.Constant(tVals[0])
poreRad = 0.1
pore_expr = d.Expression(f"(1-exp(-tCur/1.0))*exp(-(pow(x[0]-{xShift},2)+pow(x[1],2)+pow(x[2]-{zCenter},2))/(pow(poreRad,2)))", tCur = t, degree=1, poreRad=poreRad)
pore_fcn = d.interpolate(pore_expr, Vcur)
for i in range(len(tVals)):
    t.assign(tVals[i])
    pore_fcn = d.interpolate(pore_expr, Vcur)
    pore_fcn.rename("pore", "pore")
    pore_xdmf.write(pore_fcn, tVals[i])